# K8s executor 위의 Spark — 계산이 어디서 도는가

Spark Connect 서버가 `--master k8s://`로 돌 때, **노트북에서 던진 SQL이 실제로 어느 파드에서
계산되는지**를 관측한다. 접속만 확인하는 [`00-lakehouse-connect.ipynb`](00-lakehouse-connect.ipynb)와
목적이 다르다.

## 전제

```shell
kubectl scale deploy/spark-connect --replicas=1          # 평시는 0이다
kubectl get pods -l spark-role=executor                  # executor 1개가 떠야 한다
kubectl port-forward svc/spark-connect 15002:15002       # 폴백 경로를 쓸 때만
```

`.env`에 `SPARK_REMOTE`가 있으면 그 값을 쓰고, 없으면 `sc://localhost:15002`로 폴백한다.
TLS Ingress 경로를 쓰려면 `GRPC_DEFAULT_SSL_ROOTS_FILE_PATH`도 함께 필요하다
(`docs/conventions/k8s.md` §10).

## 입력 데이터

| 테이블 | 쓰는 곳 |
| --- | --- |
| `iceberg.usgs_water.water_iv_raw` | 조회·집계·파일 통계 |
| `iceberg.usgs_water.iv_joined` | 스냅샷 계보 관측 |

원천은 **USGS Instantaneous Values Web Service**(미국 지질조사국 수문 관측). 공개 데이터라
반출 제약이 없다. 개인정보·DUA 대상 데이터셋(`mimiciv`·`eicu`)은 이 노트북에서 다루지 않는다.

## 이 노트북은 읽기 전용이다

테이블을 만들거나 고치지 않는다. 정의(에셋·dbt 모델)도 두지 않는다 —
단일 출처는 `defs/`·`models/`다(`docs/conventions/analysis.md`).

## 1. 세션

🔴 `load_dotenv`를 **맨 앞에서** 부른다. 아래쪽에서 부르면 그 위의 셀은 `.env`를 못 본 채
기본값으로 접속하고, 그래도 돌기 때문에 **틀린 대상에 붙은 것을 눈치채지 못한다**.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# repo 루트의 .env를 읽어 os.environ에 주입한다(노트북 cwd = notebooks/).
ENV_PATH = Path.cwd().parent / ".env"
loaded = load_dotenv(ENV_PATH)

SPARK_REMOTE = os.environ.get("SPARK_REMOTE", "sc://localhost:15002")
print(f".env      : {ENV_PATH} ({'읽음' if loaded else '없음 — 기본값으로 간다'})")
print(f"접속 대상 : {SPARK_REMOTE}")

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
print("Spark", spark.version)

## 2. 클라이언트가 못 바꾸는 것

카탈로그·executor 설정의 **단일 출처는 서버 측 매니페스트**(`k8s/spark/spark-connect-server.yaml`)다.
노트북에서 바꾸려 하면 두 경로가 **서로 다르게** 실패한다 — 이게 헷갈리는 지점이다.

| 경로 | 결과 |
| --- | --- |
| `SparkSession.builder.config(...)` + `.remote(...)` | 조용히 무시된다(경고로 강등) |
| `spark.conf.set(...)` (런타임) | `AnalysisException: CANNOT_MODIFY_CONFIG`로 **시끄럽게** 실패 |

🔴 **"설정했다"와 "적용됐다"는 다른 축이다.** 위쪽 경로는 반환값도 예외도 없어서,
적용된 줄 알고 지나가기 쉽다. 아래 셀은 **아래쪽 경로**를 관측한다.

In [ ]:
# 런타임 경로 — 서버가 명시적으로 거부한다.
try:
    spark.conf.set("spark.executor.instances", "2")
    print("런타임 set : 예외 없음(예상 밖)")
except Exception as exc:
    print(f"런타임 set : {type(exc).__name__} — {str(exc).splitlines()[0][:80]}")

# 서버가 실제로 보고 있는 값
print("서버 실제값:", spark.conf.get("spark.executor.instances", "<못 읽음>"))
print("서버 master:", spark.conf.get("spark.master", "<못 읽음>"))

### Connect 세션이 지원하지 않는 것

`sparkContext`는 없다. RDD API에 의존하는 코드는 Connect로 그대로 옮길 수 없다
(`docs/architectures/spark.md`).

In [ ]:
try:
    _ = spark.sparkContext
    print("sparkContext 접근 성공(예상 밖)")
except Exception as exc:
    print(f"sparkContext: {type(exc).__name__} — {str(exc).splitlines()[0][:80]}")

## 3. 🔴 계산이 어디서 도는가 — 이 노트북의 급소

**executor 파드가 떠 있는 것**과 **태스크가 거기서 도는 것**은 다른 축이다.
`kubectl get pods`는 앞의 것만 말한다.

### Python UDF로는 확인할 수 없다

호스트명을 돌려주는 Python UDF를 걸면 될 것 같지만, Connect에서 Python UDF는
**클라이언트에서 직렬화돼 executor의 Python worker가 실행**한다. 두 Python의 minor 버전이
다르면 `PYTHON_VERSION_MISMATCH`로 죽는다 — 러너 이미지는 3.10, 이 노트북 커널은 3.12다.

### 그래서 Spark REST API로 센다

쿼리 **전후로 executor별 누적 태스크 수**를 읽어 증가분을 본다.
🔴 `driver` 행이 **함께 0으로 남는 것**이 음성 대조다 — executor 쪽만 보면
"원래 그 숫자였다"와 구분되지 않는다.

In [ ]:
import json
import urllib.request

# Spark UI는 Ingress로 나가 있다(kind extraPortMappings 80→8080).
# 폴백: kubectl port-forward svc/spark-connect 4040:4040 후 http://localhost:4040
UI_BASE = os.environ.get("SPARK_UI_BASE", "http://spark.localtest.me:8080")


def executor_tasks() -> dict[str, dict]:
    """Executor id → {hostPort, completedTasks} 를 REST API에서 읽는다."""
    # S310: URL은 위 상수에서만 오고 스킴이 http로 고정이라 감사 대상이 아니다.
    with urllib.request.urlopen(f"{UI_BASE}/api/v1/applications", timeout=10) as resp:  # noqa: S310
        app_id = json.load(resp)[0]["id"]
    url = f"{UI_BASE}/api/v1/applications/{app_id}/executors"
    with urllib.request.urlopen(url, timeout=10) as resp:  # noqa: S310
        return {e["id"]: e for e in json.load(resp)}


before = executor_tasks()
for eid, e in sorted(before.items()):
    print(f"  [전] {eid:8s} {e['hostPort']:22s} completedTasks={e['completedTasks']}")

In [ ]:
TABLE = "iceberg.usgs_water.water_iv_raw"

rows = spark.sql(f"SELECT count(*) AS c FROM {TABLE}").collect()[0].c
print(f"{TABLE} 행 수 = {rows:,}\n")

after = executor_tasks()
for eid, e in sorted(after.items()):
    delta = e["completedTasks"] - before.get(eid, {}).get("completedTasks", 0)
    mark = "  🔴 여기서 돌았다" if eid != "driver" and delta > 0 else ""
    tasks = f"completedTasks={e['completedTasks']} (+{delta})"
    print(f"  [후] {eid:8s} {e['hostPort']:22s} {tasks}{mark}")

터미널에서 파드 IP와 대조한다 — 위 `hostPort`의 주소가 executor 파드의 IP여야 한다.

```shell
kubectl get pods -l spark-role=executor -o wide
```

## 4. 데이터 훑기

집계는 **Spark에서** 하고 `toPandas()`는 결과에만 건다.
전량을 드라이버 메모리로 끌어오지 않는다(`docs/conventions/analysis.md`).

In [ ]:
spark.sql(f"""
    SELECT
        count(*)                  AS rows,
        count(DISTINCT site_no)   AS sites,
        min(date_time)            AS first_obs,
        max(date_time)            AS last_obs
    FROM {TABLE}
""").show(truncate=False)

## 5. 파티션의 두 얼굴

🔴 **Dagster 파티션과 Iceberg 파티션은 다른 축이다.** 이름이 같아 같은 것으로 읽힌다.

| | 무엇인가 | 어디에 있나 |
| --- | --- | --- |
| **Dagster 파티션** | 적재의 **논리 단위**(어느 조각을 다시 쓸지) | 에셋 정의 · `replace_partition_in_iceberg`의 `overwrite_filter` |
| **Iceberg 파티션** | 파일의 **물리 레이아웃**(디렉터리 분할) | 테이블의 `PARTITIONED BY` 스펙 |

이 저장소의 적재 헬퍼는 `create_table(identifier, schema=schema)`로 테이블을 만든다 —
`partition_spec`을 주지 않으므로 **Iceberg 파티션 스펙은 비어 있다.**

스펙이 비면 `.partitions`에 `partition` 컬럼 자체가 없고 테이블 전체 요약 한 줄만 나온다.
**그 「컬럼이 없다」가 곧 「스펙이 없다」의 증거**다.

In [ ]:
spark.sql(f"SELECT * FROM {TABLE}.partitions").show(truncate=False)

### 그래서 스킵은 파일 통계로 일어난다

파티션 프루닝이 아니라 **파일별 min/max 통계**(`lower_bounds`/`upper_bounds`)로
읽을 파일을 고른다. 파일 수가 적으면 효과도 작다.

In [ ]:
spark.sql(f"""
    SELECT file_path, record_count, file_size_in_bytes
    FROM {TABLE}.files
""").show(truncate=False)

## 6. 스냅샷 계보

🔴 **계보는 `.history`의 `parent_id`로 걸어야 한다.** `.snapshots`를 `committed_at`으로
정렬한 인접 두 행이 부모-자식이라는 보장이 없고, 아니면 `is not a parent ancestor`로 죽는다.

`df.writeTo(...).createOrReplace()`로 쓴 테이블은 스냅샷이 남지만 **`parent_id`가 전부 `NULL`**이라
선형 체인이 아니라 **독립 루트가 쌓인다** — 데이터는 멱등이어도 계보는 매번 끊긴다
(`docs/architectures/spark.md`). 그런 테이블은 changelog·스트리밍 소스가 될 수 없다.

⚠️ **아래 테이블은 그 사례가 아니다.** 계보가 온전한 쪽을 먼저 봐 두어야 끊긴 것을 알아본다.
판정은 `roots`(부모 없는 스냅샷)로 한다 — **1이면 선형**, 여럿이면 쓰기마다 끊긴 것이다.

In [ ]:
LINEAGE_TABLE = "iceberg.usgs_water.iv_joined"

spark.sql(f"""
    SELECT
        made_current_at,
        snapshot_id,
        parent_id,
        is_current_ancestor
    FROM {LINEAGE_TABLE}.history
    ORDER BY made_current_at DESC
""").show(10, truncate=False)

In [ ]:
# roots = 부모 없는 스냅샷 수. 1이면 선형 체인, 여럿이면 쓰기마다 계보가 끊긴 것이다.
spark.sql(f"""
    SELECT
        count(*)                                            AS snapshots,
        count(parent_id)                                    AS with_parent,
        count(*) - count(parent_id)                         AS roots,
        sum(CASE WHEN is_current_ancestor THEN 1 ELSE 0 END) AS current_ancestors
    FROM {LINEAGE_TABLE}.history
""").show(truncate=False)

## 7. 정리

Spark Connect는 **상주 컴퓨트**이고, `k8s://` 전환 이후에는 **executor 파드도 함께 상주**한다
(`spark.executor.instances`는 정적 할당이라 세션이 살아 있는 내내 산다).

```shell
kubectl scale deploy/spark-connect --replicas=0
kubectl get pods -l spark-role=executor       # 🔴 executor도 함께 사라져야 한다
```

🔴 **executor가 남으면 전환이 미완성이다.** driver 파드를 소유자로 걸어 두지 않으면
(`spark.kubernetes.driver.pod.name`) `--replicas=0`이 driver만 내리고 executor는 남는데,
에러도 알림도 없이 1 CPU를 계속 점유한다. 회수 규율은 예산이 늘어도 유지된다
(`docs/conventions/k8s.md` §9-3).

In [ ]:
spark.stop()